# Framework + R0 Baseline

This notebook documents the shared repair framework and the R0 baseline, with runnable cells that exercise the real code. Everything here runs **offline** (no paid API calls, no network), so it reproduces on any machine. It covers:

1. Environment / data availability check.
2. English query loading from the bilingual Hugging Face dataset.
3. The unit-test suite (pass/fail counts, with known environment errors explained).
4. The shared verifier bucketing genuine violations vs. technical/structural errors.
5. Personal constraints (`hard_logic_py`) evaluated directly.
6. An end-to-end R0 repair loop driven by a **mock LLM**.

The real end-to-end LLM runs aren't re-run here (they need an API key, cost money, and aren't reproducible); their artifacts are kept under `cache/` and referenced where relevant. The full R0 experiment on the train split (using DeepSeek V3) is in the companion notebook `train_test_split_r0.ipynb`.

> **Environment:** Python 3.12.8, with both the English (`database_en`) and Chinese (`database`) sandboxes installed via `scripts/setup_project.py`. **English is the default** (`--lang en`), so queries, cities, and output are all English, and the proof cells use English data.

## Step 0: Environment & Data Check

First confirm the Python version, the pinned dependencies, which sandbox datasets are present, and whether the API keys are configured. The API key values are **never** printed — only whether the variable name exists in `.env`.

In [1]:
import os, platform

# Run from the repository root so `chinatravel` imports resolve.
if os.path.basename(os.getcwd()) != 'Agentic-Travel-Planner-Ablation-of-Verifications-' \
        and os.path.isdir('Agentic-Travel-Planner-Ablation-of-Verifications-'):
    os.chdir('Agentic-Travel-Planner-Ablation-of-Verifications-')

print('Python:', platform.python_version())
for mod in ['numpy', 'pandas', 'pyarrow', 'jsonschema']:
    try:
        m = __import__(mod)
        import importlib.metadata as md
        print(f'  {mod}:', md.version(mod))
    except Exception as e:
        print(f'  {mod}: NOT INSTALLED ({e})')

print('Chinese sandbox (database)    present:', os.path.isdir('chinatravel/environment/database'))
print('English sandbox (database_en) present:', os.path.isdir('chinatravel/environment/database_en'))

# Report API-key configuration by NAME only, never the value.
if os.path.exists('.env'):
    names = {l.split('=')[0].strip() for l in open('.env')
             if '=' in l and not l.strip().startswith('#')}
    for k in ['OPENROUTER_API_KEY', 'CHINATRAVEL_OPENAI_API_KEY']:
        print(f'  {k} configured in .env:', k in names)

Python: 3.12.8
  numpy: 1.26.4


  pandas: 2.2.3
  pyarrow: 19.0.1
  jsonschema: 4.23.0
Chinese sandbox (database)    present: True
English sandbox (database_en) present: True
  OPENROUTER_API_KEY configured in .env: True
  CHINATRAVEL_OPENAI_API_KEY configured in .env: True


### Proof: English queries load end-to-end

The ChinaTravel query files on Hugging Face are **bilingual** — each query carries a Chinese (`nature_language`) and an English (`nature_language_en`) version. With `--lang en`, the loader downloads the same dataset, uses the English query text, and translates the city names to English (via the canonical `CITY_NAMES` map) so they resolve against the English sandbox (`database_en`). This confirms an **English input** is what reaches the agent, so the generated itinerary is produced in English.


In [2]:
import io, contextlib
from types import SimpleNamespace
from chinatravel.data.load_datasets import load_query

# Load the 'medium' split in English straight from the bilingual HF dataset.
args = SimpleNamespace(splits='medium', lang='en', oracle_translation=True)
with contextlib.redirect_stdout(io.StringIO()):
    ids, data = load_query(args)

print(f'Loaded {len(ids)} English queries from the "medium" split.')
q = data['e20241028160845920703']
print('Example query (English):')
print('  start_city :', q['start_city'])
print('  target_city:', q['target_city'])
print('  request    :', q['nature_language'][:150])
print('  constraints:', q['hard_logic_py'][:2], '...')

Loaded 150 English queries from the "medium" split.
Example query (English):
  start_city : Shenzhen
  target_city: Guangzhou
  request    : Current location: Shenzhen. I plan to visit Guangzhou for two days with a budget of 1000 RMB. I want to eat hotpot and book a single-bed room. Please 
  constraints: ['result=(day_count(plan)==2)', 'result=(people_count(plan)==1)'] ...


## Step 1: How the original repair loop works

The original ChinaTravel agent lives in `chinatravel/agent/nesy_verifier/llm_modulo.py`; the main method is `LLMModuloAgent.solve(problem, prob_idx, oracle_verifier)`. It has no dedicated repair function — it "repairs" a plan by re-prompting the LLM with the previous plan plus the verifier's feedback.

**Flow:**

1. **Collect information:** shortlist transport, hotels, attractions and restaurants from the environment.
2. **Generate:** use `INITIAL_PROMPT` to produce the initial itinerary (step 0).
3. **Verify:** run `collect_commonsense_constraints_error` and `collect_personal_error` to find violations.
4. **Repair:** if there are errors, use `BACK_PROMPT` to regenerate using the previous plan and feedback.
5. **Repeat:** until no errors are reported or the repair limit is reached.
6. **Return:** the final itinerary and verification results.

**Key points:**

- `max_steps` is the repair-attempt limit, set by `--refine_steps` (default 10).
- R0 regenerates the *entire* itinerary on repair; it doesn't protect already-satisfied constraints.
- `error_info` lists reported violations; an empty list means nothing was reported, though some checks may have been skipped.
- `solve()` returning `True` does not guarantee all constraints passed.

## Step 2: Setting Up and Testing R0

### 2.1 Environment Setup

Set up ChinaTravel using Python 3.12.8 and the dependencies in `requirements.txt`.

**Configuration:**

- **Dataset:** ChinaTravel dataset from Hugging Face
- **Sandbox:** ChinaTravel-Sandbox. Both the English (`database_en`) and Chinese (`database`) sandboxes are installed; **English is the default** language for our experiments (`--lang en`).
- **LLM:** DeepSeek V3 (`deepseek/deepseek-chat`) through OpenRouter (low-cost; see notes). Earlier validation runs used GPT-4.1.
- **Baseline:** LLM-Modulo (R0)

**Issues encountered and fixes:**

1. **Dependencies:** Used NumPy 1.26.4 and PyArrow 19.0.1. Fixed the missing `_lzma` module by reinstalling Python with XZ support.
2. **Dataset format:** `hard_logic_py` was stored as a string instead of a list. Updated `load_datasets.py` to convert it using `ast.literal_eval()`.
3. **Missing sandbox:** Downloaded the official ChinaTravel-Sandbox. `scripts/setup_project.py` installs both the English (`database_en`) and Chinese (`database`) sandboxes.
4. **Missing import:** Added `import sys` to `llm_modulo.py`.
5. **OpenRouter:** Updated the model configuration to support `openai/gpt-4.1`.
6. **English as default:** The run/eval scripts (`run_exp.py`, `run_tpc.py`, `eval_exp.py`, etc.) now default to `--lang en`, so the interface and all produced output are in English unless `--lang zh` is passed explicitly.

`scripts/setup_project.py` automates sandbox installation and `.env.example` documents the API configuration, so teammates can follow **Quick Setup** in `README.md` instead of repeating these steps.

### 2.2 Testing the original R0 (real GPT-4.1 runs)

The original R0 was run with `--refine_steps 2`. These used the real API, so the evidence is the saved run artifacts under `cache/` rather than live output.

**Test 1 — Easy split.** Case `e20241028160248698752`: one-day Shanghai→Hangzhou trip, budget 1,500 RMB. The initial itinerary passed the reported checks; no repair was needed (`ITER 0 ERROR: []`).

**Test 2 — Medium split.** Case `e20241028160845920703`: two-day Shenzhen→Guangzhou trip, budget 1,000 RMB, hotpot preference and single-bed room. The initial itinerary violated a dinner-timing constraint; R0 regenerated the itinerary and after one repair the verifier returned `ITER 1 ERROR: []`. Personal constraints were not fully checked here because `hard_logic_nl` was missing — one of the motivations for the shared verifier below.

The cell below lists the saved artifacts from these runs as evidence.

In [3]:
import os, glob

# Saved artifacts from real R0 runs (prompts, plans, errors, logs). Matches the
# GPT-4.1 cache dirs (old nested + sanitized flat) and the DeepSeek dirs.
roots = (glob.glob('cache/*oracletranslation*/llm_modulo-openai/gpt-4.1')
         + glob.glob('cache/*oracletranslation*/llm_modulo-openai_gpt-4.1')
         + glob.glob('cache/*oracletranslation*/llm_modulo-deepseek_deepseek-chat'))
if not roots:
    print('No cached run artifacts found under cache/.')
for root in roots:
    print('Run cache:', root)
    files = sorted(os.listdir(root))
    for f in files[:12]:
        print('   ', f)
    if len(files) > 12:
        print(f'    ... and {len(files) - 12} more files')

Run cache: cache/LLM-modulo_openai_gpt-4.1_2steps_oracletranslation_R1/llm_modulo-openai/gpt-4.1
Run cache: cache/LLM-modulo_openai_gpt-4.1_2steps_oracletranslation_R0/llm_modulo-openai/gpt-4.1
    problem_e20241028160845920703.error
    problem_e20241028160845920703.log
    problem_e20241028160845920703_repair_history.json
    problem_e20241028160845920703_step_0_prompt.txt
Run cache: cache/LLM-modulo_openai_gpt-4.1_en_5steps_oracletranslation_R0/llm_modulo-openai/gpt-4.1
    problem_e20241028160255878579.error
    problem_e20241028160255878579.log
    problem_e20241028160255878579_repair_history.json
    problem_e20241028160255878579_step_0_prompt.txt
    problem_e20241028160300570125.error
    problem_e20241028160300570125.log
    problem_e20241028160300570125_repair_history.json
    problem_e20241028160300570125_step_0_prompt.txt
    problem_e20241028160848776495.error
    problem_e20241028160848776495.log
    problem_e20241028160848776495_repair_history.json
    problem_e202410281

### 2.3 Logging

The original ChinaTravel implementation saves files for each iteration:

| File | Purpose |
|---|---|
| `step_0_prompt.txt` | Initial generation prompt |
| `step_0_plan.json` | Initial itinerary and verification results |
| `step_0_error.err` | Initial verifier feedback |
| `step_1_...` | Corresponding files after repair |

The saved plans also contain token usage and execution time. The original logs don't track constraint transitions; that's added in `shared_repair_loop.py`, which records which personal constraints were fixed, newly violated, still failed or still satisfied (demonstrated in Step 3).

## Step 3: The shared framework

The repair process is pulled out of the original agent so R0–R4 share one planner, verifier and repair loop. A new strategy only needs to implement its `repair()` method.

### 3.1 Files added

All in `chinatravel/agent/nesy_verifier/`:

| File | Purpose |
|---|---|
| `shared_verifier.py` | Checks commonsense and personal constraints |
| `shared_repair_loop.py` | Handles repair attempts, verification, accept/reject and logging |
| `repair_strategies/base.py` | Common interface for R0–R4 |
| `repair_strategies/r0.py` | R0 whole-plan regeneration |

`run_exp.py`, `load_model.py` and `llm_modulo.py` were updated to support selecting a repair strategy.

### 3.2 How it works

1. ChinaTravel generates the initial itinerary.
2. The shared verifier checks commonsense and personal constraints.
3. The selected strategy proposes a revised itinerary.
4. The shared verifier checks the proposal.
5. The strategy decides whether to accept or reject it.
6. Repeat until verification passes or the repair limit is reached.

The original R0 loop still runs when no strategy is selected.

### 3.3 Improvements

- **Shared verification:** uses `hard_logic_py` directly, even when `hard_logic_nl` is missing.
- **Raw itinerary preserved:** keeps the original itinerary separate from the translated-transport version.
- **Accept/reject:** strategies can reject a proposal and keep the previous plan.
- **Constraint-transition logging:** records which personal constraints were fixed, newly violated, still failed, still passed, or couldn't be evaluated.
- **Verifier reliability:** separates genuine violations from technical errors (verifier exceptions, translation failures, bad structure).
- **Separate outputs:** each strategy gets its own result and log directories.

The framework doesn't auto-repair, protect constraints, or reject harmful changes — those decisions belong to each R0–R4 strategy.

### 3.4 Proof: Unit-test suite

Run the full test suite for the shared framework and evaluation code. The cell reports the real counts and prints any errors.

**Expected:** with both the Chinese and English sandboxes installed (via `scripts/setup_project.py`), the full suite passes with no failures and no errors. If the English sandbox (`database_en`) is missing, three data-dependent tests would surface as *errors* (missing data file) rather than genuine *failures*; the cell distinguishes the two so a missing-data situation is never mistaken for a code defect.

In [4]:
import unittest, io, contextlib, os

# The test suite prints its own progress/logging to stdout; redirect it so
# only the clean summary below is shown. The result object is unaffected.
loader = unittest.TestLoader()
suite = loader.discover('tests', pattern='test_*.py')
buf = io.StringIO()
with open(os.devnull, 'w') as devnull, contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
    result = unittest.TextTestRunner(stream=buf, verbosity=0).run(suite)

print(f'Tests run : {result.testsRun}')
print(f'Failures  : {len(result.failures)}')
print(f'Errors    : {len(result.errors)}')
print(f'Passed    : {result.testsRun - len(result.failures) - len(result.errors)}')
print()
if result.errors:
    print('Errors (with both sandboxes installed, expect none; any here are')
    print('typically missing-data / environment issues, not code defects):')
    for test, _ in result.errors:
        print('   -', test.id())
if result.failures:
    print('Genuine failures:')
    for test, _ in result.failures:
        print('   -', test.id())
else:
    print('No genuine test failures.')

Tests run : 335
Failures  : 0
Errors    : 0
Passed    : 335

No genuine test failures.


### Proof: Shared verifier buckets errors correctly

We feed the verifier a deliberately **broken** itinerary: a standalone `innercity_transport` activity (which should be nested inside a destination activity's `transports` field) and an activity missing `start_time`. The verifier should:

- flag the structural problems in `schema_errors`,
- keep the "no translation function provided" notice in `translation_errors`,
- report genuine commonsense violations in `commonsense_errors`,
- and mark the plan `passed = False`.

This shows technical/structural problems are reported on their own channels and never silently pass.

In [5]:
from chinatravel.agent.nesy_verifier.shared_verifier import verify_plan

# English data: Shanghai -> Suzhou, with 'Humble Administrator's Garden'
# (the English sandbox name). Structural checks are language-independent; the
# commonsense verifier resolves the English city/POI against database_en.
problem = {'people_number': 2, 'start_city': 'Shanghai', 'target_city': 'Suzhou', 'hard_logic_py': []}

broken_plan = {
    'people_number': 2, 'start_city': 'Shanghai', 'target_city': 'Suzhou',
    'itinerary': [
        {'day': 1, 'activities': [
            # WRONG: inner-city transport as a standalone activity.
            {'type': 'innercity_transport', 'start_time': '08:00', 'end_time': '08:20'},
            # WRONG: attraction missing start_time.
            {'type': 'attraction', 'position': "Humble Administrator's Garden", 'end_time': '11:00'},
        ]},
    ],
}

import io, contextlib
# Suppress the verifier's incidental stdout so only the result buckets show.
with contextlib.redirect_stdout(io.StringIO()):
    res = verify_plan(problem, broken_plan, translate_fn=None)
print('passed            :', res['passed'])
print('schema_errors     :', len(res['schema_errors']))
for e in res['schema_errors']:
    print('    -', e)
print('translation_errors:', res['translation_errors'])
print('commonsense_errors:', len(res['commonsense_errors']))
print('verifier_errors   :', len(res['verifier_errors']))

passed            : False
schema_errors     : 2
    - Day 1 activity 1 has unsupported type 'innercity_transport'; inner-city transport must be nested in a destination activity's 'transports' field, not a separate activity.
    - Day 1 activity 2 ('attraction') is missing 'start_time'.
translation_errors: ['No transport translation function provided.']
commonsense_errors: 7
verifier_errors   : 0


### Proof: Personal constraints (`hard_logic_py`) evaluated directly

The shared verifier evaluates each `hard_logic_py` expression against the plan and returns `True` (pass), `False` (fail) or `None` (could not be evaluated). Here we pass one constraint that should hold and one that should not.

In [6]:
from chinatravel.agent.nesy_verifier.shared_verifier import check_personal_constraints

good_plan = {
    'people_number': 2, 'start_city': 'Shanghai', 'target_city': 'Suzhou',
    'itinerary': [
        {'day': 1, 'activities': [
            {'type': 'attraction', 'position': "Humble Administrator's Garden",
             'start_time': '09:00', 'end_time': '11:00', 'transports': 'empty'},
        ]},
    ],
}
prob_pc = {
    'people_number': 2, 'start_city': 'Shanghai', 'target_city': 'Suzhou',
    'hard_logic_py': [
        "result=(plan['people_number']==2)",   # expect True
        "result=(plan['people_number']==5)",   # expect False
    ],
}
results, eval_errors = check_personal_constraints(prob_pc, good_plan)
print('per-constraint results (True=pass, False=fail, None=uneval):', results)
print('evaluation_errors:', eval_errors)

per-constraint results (True=pass, False=fail, None=uneval): [True, False]
evaluation_errors: []


### Proof: End-to-end R0 repair loop (mock LLM, no API cost)

We run the real `run_repair_loop` with the real `R0WholePlanRepair` strategy, but drive it with a **mock LLM** so no network/API call is made. This proves the loop wiring end-to-end: it verifies the initial plan, calls the strategy once per repair step, re-verifies, applies the accept/reject decision, and records a step-by-step history.

The history table below is the **constraint-transition logging** in action: each step records the accept decision, the `passed` flag, and the per-constraint results.

In [7]:
from chinatravel.agent.nesy_verifier.repair_strategies.r0 import R0WholePlanRepair
from chinatravel.agent.nesy_verifier.shared_repair_loop import run_repair_loop

start_itinerary = [
    {'day': 1, 'activities': [
        {'type': 'attraction', 'position': "Humble Administrator's Garden",
         'start_time': '09:00', 'end_time': '11:00', 'transports': 'empty'},
    ]},
]
prob_e2e = {
    'people_number': 2, 'start_city': 'Shanghai', 'target_city': 'Suzhou',
    'nature_language': 'demo query',
    'hard_logic_py': ["result=(len(plan['itinerary'])>=1)"],
}
context = {k: '' for k in [
    'go_train_info', 'go_flight_info', 'back_train_info', 'back_flight_info',
    'attraction_info', 'restaurant_info', 'accommodation_info', 'innercity_transport_info',
]}

class MockLLM:
    calls = 0
    def __call__(self, messages, one_line=False):
        MockLLM.calls += 1
        # R0 prepends '[{"day": 1,' so we return the rest of a valid 1-day plan.
        return (' "activities": [{"type": "attraction", "position": "Humble Administrator\'s Garden",'
                ' "start_time": "09:00", "end_time": "11:00", "transports": "empty"}]}]')

def identity_translate(itinerary, problem):
    return itinerary  # offline stand-in for transport translation

import io, contextlib
# Suppress the verifier's incidental stdout; the history table below is the proof.
with contextlib.redirect_stdout(io.StringIO()):
    final = run_repair_loop(
        problem=prob_e2e, raw_plan=start_itinerary, strategy=R0WholePlanRepair(),
        context=context, llm=MockLLM(), translate_fn=identity_translate, max_steps=2,
    )

print('mock LLM calls         :', MockLLM.calls)
print('history steps recorded :', len(final['history']))
print()
print(f"{'step':>4} | {'accepted':>8} | {'passed':>6} | personal | schema_errors")
print('-' * 54)
for r in final['history']:
    v = r.get('verification', {})
    print(f"{r.get('step'):>4} | {str(r.get('accepted')):>8} | "
          f"{str(v.get('passed')):>6} | {str(v.get('personal_constraints')):>8} | "
          f"{len(v.get('schema_errors', []))}")

mock LLM calls         : 1
history steps recorded : 2

step | accepted | passed | personal | schema_errors
------------------------------------------------------
   0 |     True |  False |   [True] | 0
   1 |     True |  False |   [True] | 0


### 3.5 Verifier reliability improvements

Three issues in the verifier could quietly distort results. Each is handled in `shared_verifier.py` by adding detection *on top of* the original verifier and strategies (the originals are untouched):

1. **Commonsense verifier exceptions.** The original verifier sometimes returned Python exceptions as normal error messages (e.g. `KeyError: 'position'`). `_split_commonsense_errors()` moves these into `verifier_errors`, leaving genuine violations in `commonsense_errors`.
2. **Silent translation failures.** Transport translation can silently replace a failed result with an empty list. `_detect_silent_translation_failures()` compares transport info before/after and records failures in `translation_errors`.
3. **Invalid itinerary structure.** The LLM sometimes emits standalone `innercity_transport` activities instead of nesting transport in the destination. `_validate_plan_structure()` flags unsupported types, missing fields and bad structure in `schema_errors`.

These checks only *detect and report* — they don't auto-fix plans or change the original constraint rules. (The error-bucketing cell above exercises checks 2 and 3.)

### 3.6 Testing after the reliability work

`tests/test_shared_verifier_reliability.py` covers the new checks and runs with the suite in 3.4. A real GPT-4.1 rerun of the medium case `e20241028160845920703` (two repairs) confirmed: genuine violations stay detectable; technical/structural errors go on their own channels; an invalid plan is still marked invalid; R0 keeps accepting proposals without protecting satisfied constraints; and constraint-transition logging catches newly introduced violations (e.g. the accommodation room-count constraint passed initially, then failed after the first repair and stayed failed). This shows how whole-plan regeneration can introduce new violations while fixing others — a key motivation for R1–R4.

**Status:** the framework and R0 pass the offline integration and reliability tests here. R1–R4 and the cross-strategy comparison are the next steps.

### 3.7 Known bugs (fixed) and where the experiment lives

A few issues were found and fixed so the pipeline runs correctly and cross-platform:

- **Empty itineraries (httpx clash):** the newer `openai` package bundles its own `httpx` (`httpx2`), but the usage-instrumentation transport in `chinatravel/agent/model_usage.py` was built against the old `httpx`, so every model request failed silently. Fixed by building the transport against whichever `httpx` the installed `openai` client uses.
- **Verifier language mismatch:** the commonsense verifier defaulted to the Chinese sandbox even for English runs, producing false "unavailable transport" errors. It is now language-aware (infers the language per query and loads the matching sandbox), so only genuine violations remain.
- **Windows logging:** long log paths (>260 chars) and un-created nested directories broke logging on Windows. Fixed with parent-dir creation and the `\\?\` extended-length path prefix (no-op on macOS/Linux).

The offline suite stays green (277 tests). The full R0 experiment — the train/test split, running R0 per query on **DeepSeek V3**, and the committed results in `experiments/r0_deepseek-chat_pilot5/` — is in `train_test_split_r0.ipynb`. Those are historical results. For new registered-strategy studies use `run_experiments.py --strategy R0 --splits pilot5 --max-steps 6 --lang en --llm YOUR_PROVIDER_MODEL_ID --run-id pilot5-r0`, followed by the independent offline stages in `experiments/Commands_to_test_queries.md` and `chinatravel/evaluation/REPAIR_COMPARISON.md`. Use `run_exp.py` for general ChinaTravel agents and legacy baselines. R0 scores a low pass rate because it lets the model write transport details and the model guesses some wrong (e.g. a metro fare), which the verifier rejects — the expected weakness of the naive baseline and the gap R1–R4 target.

## Step 4: Getting started (for teammates implementing R1–R4)

### 4.1 Setup

The framework is on `main`. Follow **Quick Setup** in `README.md`: install dependencies, run `scripts/setup_project.py` for the sandbox, add your OpenRouter key to `.env` (see `.env.example`), and run one R0 case to check the environment.

### 4.2 Run flags

- splits: easy, medium subset of queries
- Index: run just a single query by its uid; default `None` runs every query in the split
- Refine steps: number of repair attempts before stopping
- Oracle-translation: the machine-checkable personal constraints are on

## How a query looks like


## Smoke run: R0 on 2 `subset_train` queries (fixed verifier)

Before the full 60-query train runs, this is a 2-query R0 smoke test on `subset_train`
to confirm the pipeline produces matrix-ready records on the **current fixed verifier**
(after the bilingual category-label fix and the transport-shape fix).

It runs the 4-step workflow from `experiments/Commands_to_test_queries.md`:
**(1)** run the strategy → **(2)** benchmark eval → **(3)** summaries → **(4)** regression matrix,
then checks that `attempts.jsonl` carries the fields the matrix needs
(`target_constraint_indices`, `before/after_personal_constraints`, `fixed`, `newly_violated`).

The two queries are one `easy` and one `human` UID drawn from `subset_train`. This makes
real DeepSeek V3 calls, so it is run once; set `RUN_SMOKE=True` to execute.

In [8]:
import os, sys, subprocess, json, glob

# Run from repo root so chinatravel imports + scripts resolve.
if os.path.basename(os.getcwd()) != 'Agentic-Travel-Planner-Ablation-of-Verifications-' \
        and os.path.isdir('Agentic-Travel-Planner-Ablation-of-Verifications-'):
    os.chdir('Agentic-Travel-Planner-Ablation-of-Verifications-')

# Use THIS kernel's interpreter (the venv) for subprocesses, not a bare 'python'
# that may resolve to a different environment without the dependencies.
PY = sys.executable

# Load .env (OpenRouter key) into the environment for the subprocess calls.
if os.path.exists('.env'):
    for line in open('.env'):
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

RUN_SMOKE = False           # outputs below are the recorded smoke run; set True to re-run (paid API calls)
RUN_ID = 'subset_train-r0-smoke'
RUN_DIR = f'experiments/{RUN_ID}'
UIDS = ['e20241028160320866976', 'h20241029143502525564']  # 1 easy + 1 human from subset_train
print('python:', PY)
print('RUN_SMOKE:', RUN_SMOKE, '| RUN_ID:', RUN_ID)
print('UIDs:', UIDS)

python: /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/.venv/bin/python
RUN_SMOKE: True | RUN_ID: subset_train-r0-smoke
UIDs: ['e20241028160320866976', 'h20241029143502525564']


### Step 1 — run R0 on the 2 train queries

In [9]:
cmd = [PY, 'run_experiments.py',
       '--strategy', 'R0', '--splits', 'subset_train',
       '--index', *UIDS,
       '--llm', 'deepseek/deepseek-chat', '--lang', 'en',
       '--max-steps', '6',
       '--output-dir', 'experiments', '--run-id', RUN_ID]
print('python run_experiments.py --strategy R0 --splits subset_train \\\n  --index ' + ' '.join(UIDS) + ' \\\n  --llm deepseek/deepseek-chat --lang en --max-steps 6 \\\n  --output-dir experiments --run-id ' + RUN_ID)
if RUN_SMOKE:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print('--- stdout (tail) ---')
    print('\n'.join(r.stdout.splitlines()[-6:]))
    if r.returncode != 0:
        print('--- stderr (tail) ---')
        print('\n'.join(r.stderr.splitlines()[-12:]))
    print('--- exit code:', r.returncode, '---')
else:
    print('skipped (RUN_SMOKE=False)')

python run_experiments.py --strategy R0 --splits subset_train \
  --index e20241028160320866976 h20241029143502525564 \
  --llm deepseek/deepseek-chat --lang en --max-steps 6 \
  --output-dir experiments --run-id subset_train-r0-smoke


--- stdout (tail) ---
Commonsense constraints failed!
Commonsense constraints failed!
Commonsense constraints failed!
Commonsense constraints failed!
Commonsense constraints failed!
Comparison outputs: experiments/subset_train-r0-smoke
--- exit code: 0 ---


### Steps 2–4 — benchmark eval, summaries, regression matrix (offline, no API)

In [10]:
steps = [
    [PY, 'scripts/evaluate_repair_run.py', '--run-dir', RUN_DIR],
    [PY, 'scripts/build_repair_summary.py', '--input', RUN_DIR],
    [PY, 'scripts/build_regression_matrix.py',
     '--input', f'{RUN_DIR}/attempts.jsonl',
     '--output-dir', f'{RUN_DIR}/regression_matrix',
     '--repair-strategy', 'R0'],
]
for s in steps:
    print('$', 'python', ' '.join(s[1:]))
    if RUN_SMOKE:
        r = subprocess.run(s, capture_output=True, text=True)
        out = (r.stdout + r.stderr).strip().splitlines()
        print('  ', '\n   '.join(out[-4:]) or '(no output)')
        print('   exit', r.returncode)
    else:
        print('  skipped')

$ python scripts/evaluate_repair_run.py --run-dir experiments/subset_train-r0-smoke


   
   100%|████████████████████████████████████████████| 2/2 [00:00<00:00,  8.80it/s]
   exit 0
$ python scripts/build_repair_summary.py --input experiments/subset_train-r0-smoke
   R0: 2 queries | pass rate step 5 0.0% -> final 0.0% | all personal pass step 5 50.0% -> final 50.0% | broken after step 5: 0/1
     commonsense step: 2 queries | errors 3 -> fixed 0, new 0 (fix rate 0.0%) | pass right after 0.0% [steps 1-2: 1 (0.0% pass), steps 3-4: 0 (n/a pass), steps 5-6: 1 (0.0% pass)] | broke personal: 0, recovered 0
   exit 0
$ python scripts/build_regression_matrix.py --input experiments/subset_train-r0-smoke/attempts.jsonl --output-dir experiments/subset_train-r0-smoke/regression_matrix --repair-strategy R0


   (no output)
   exit 0


### Verify: records are matrix-ready on the fixed verifier

Each attempt should carry `target_constraint_indices` and tri-state
`before/after_personal_constraints`, and the regression-matrix CSVs should exist.
If the old bugs were still live, personal constraints would mostly be `None`
(unevaluable) and the matrix would have nothing to learn from.

In [11]:
records = [json.loads(l) for l in open(f'{RUN_DIR}/attempts.jsonl')] \
          if os.path.exists(f'{RUN_DIR}/attempts.jsonl') else []
print('attempt records:', len(records))
none_total = ok_total = 0
for r in records:
    after = r.get('after_personal_constraints') or []
    none_total += sum(1 for v in after if v is None)
    ok_total += sum(1 for v in after if v is not None)
    print(f"  step {r.get('step')} uid {str(r.get('query_uid'))[:14]} "
          f"target={r.get('target_constraint_indices')} "
          f"fixed={r.get('fixed')} newly_violated={r.get('newly_violated')} "
          f"accepted={r.get('accepted')}")
print(f"\npersonal-constraint results: evaluable={ok_total}, unevaluable(None)={none_total}")
mtx = sorted(glob.glob(f'{RUN_DIR}/regression_matrix/*.csv'))
print('regression matrix CSVs:', len(mtx))
for m in mtx[:6]:
    print('  ', os.path.basename(m))
print('\nSMOKE OK:' , bool(records) and ok_total > 0 and len(mtx) > 0)

attempt records: 7
  step 1 uid e2024102816032 target=[] fixed=[] newly_violated=[] accepted=True
  step 1 uid h2024102914350 target=[3] fixed=[] newly_violated=[] accepted=True
  step 2 uid h2024102914350 target=[3] fixed=[] newly_violated=[] accepted=True
  step 3 uid h2024102914350 target=[3] fixed=[] newly_violated=[] accepted=True
  step 4 uid h2024102914350 target=[3] fixed=[] newly_violated=[] accepted=True
  step 5 uid h2024102914350 target=[3] fixed=[] newly_violated=[] accepted=True
  step 6 uid h2024102914350 target=[] fixed=[] newly_violated=[] accepted=True

personal-constraint results: evaluable=41, unevaluable(None)=0
regression matrix CSVs: 11
   accepted_regression_count.csv
   accepted_regression_rate.csv
   accepted_regression_support.csv
   candidate_regression_count.csv
   candidate_regression_rate.csv
   candidate_regression_support.csv

SMOKE OK: True
